# 10 — Silver Reconciliation

Compares Bronze source counts with Silver clean counts and quarantine counts.

For each dataset:
- Bronze count = source rows.
- Silver count = accepted clean rows.
- Quarantine count = rejected rows.
- Expected reconciliation: Bronze count = Silver count + Quarantine count.

This count reconciliation is a basic control. It does not prove business correctness or detect every possible source-system issue.
Run all Silver processing notebooks before this notebook.


In [ ]:
from pyspark.sql import functions

spark.sql("USE CATALOG Capstone_team01")

table_pairs = [
    ("card_transactions", "card_transactions", "quarantine_card_transactions"),
    ("device_authentication_events", "device_authentication_events", "quarantine_device_authentication_events"),
    ("payment_status_changes", "payment_status_changes", "quarantine_payment_status_changes"),
    ("customer_master", "customer_master", "quarantine_customer_master"),
    ("merchant_master", "merchant_master", "quarantine_merchant_master"),
    ("card_master", "card_master", "quarantine_card_master")
]

results = []

for bronze_name, silver_name, quarantine_name in table_pairs:
    bronze_count = spark.table("Capstone_team01.bronze." + bronze_name).count()
    silver_count = spark.table("Capstone_team01.silver." + silver_name).count()
    quarantine_count = spark.table("Capstone_team01.silver." + quarantine_name).count()

    reconciled = bronze_count == silver_count + quarantine_count

    results.append((
        bronze_name,
        bronze_count,
        silver_count,
        quarantine_count,
        reconciled
    ))

reconciliation_df = spark.createDataFrame(
    results,
    ["dataset", "bronze_count", "silver_count", "quarantine_count", "is_reconciled"]
)

display(reconciliation_df)

(
    reconciliation_df
    .withColumn("checked_at", functions.current_timestamp())
    .write.format("delta").mode("overwrite")
    .saveAsTable("Capstone_team01.silver.silver_reconciliation")
)
